# Cointegrated system

N assets driven by K common random-walk factors plus stationary OU spreads: `log p = log p0 + loadings @ f + s`. With `K < N` and full-rank loadings there are exactly `N - K` cointegrating relations.

Fantasma simulates **log returns** (log price differences): one row per step, labelled at the end of the step. `fm.to_prices`, `fm.to_log_prices` and `fm.to_returns` turn them into other quantities.

Edit the parameters cell and re-run all: the same parameters and seed always give the same paths.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import fantasma as fm

## Parameters

Model parameters are annualised; `dt` is the step length over a 365.25-day year.

In [ ]:
SEED = 42
N_PATHS = 5
START = "2024-01-01"
FREQ = "1d"  # Almanac frequency ("1m", "10m", "1h", "1d") or any pandas offset
N_STEPS = 365 * 5

ASSETS = ["X", "Y", "Z"]
LOADINGS = [[1.0], [0.5], [2.0]]  # (N, K)
SPREAD_HALF_LIVES = [0.05, 0.1, 0.2]  # years, one per asset
FACTOR_SIGMAS = 0.3  # one per factor, or a scalar
SPREAD_SIGMAS = 0.1  # one per asset, or a scalar
INITIAL_PRICES = [100.0, 50.0, 200.0]

## Simulate

In [ ]:
model = fm.Cointegrated(
    loadings=LOADINGS,
    spread_half_lives=SPREAD_HALF_LIVES,
    factor_sigmas=FACTOR_SIGMAS,
    spread_sigmas=SPREAD_SIGMAS,
    assets=ASSETS,
    initial_prices=INITIAL_PRICES,
)
model

In [ ]:
paths = fm.simulate(
    model,
    start=START,
    freq=FREQ,
    n_steps=N_STEPS,
    seed=SEED,
    n_paths=N_PATHS,
    return_hidden=True,
)
paths = paths if N_PATHS > 1 else [paths]  # one path is returned itself, not in a list
log_returns, hidden = paths[0]  # log returns; hidden state on the full grid
log_returns.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for i, (path_log_returns, _) in enumerate(paths):
    path_prices = fm.to_prices(path_log_returns)  # start included, from the recorded start price
    for asset in path_prices.columns:
        ax.plot(path_prices.index, path_prices[asset], lw=0.8, label=f"path {i} {asset}")
ax.set_title("Cointegrated system")
ax.set_ylabel("price")
if len(paths) * paths[0][0].shape[1] <= 10:
    ax.legend(fontsize=8)
plt.show()

## Hidden state

The factors and spreads of path 0.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
factors = [c for c in hidden.columns if c.startswith("factor_")]
spreads = [c for c in hidden.columns if c.startswith("spread_")]
hidden[factors].plot(ax=ax1, lw=0.8, title="factors")
hidden[spreads].plot(ax=ax2, lw=0.8, title="spreads")
plt.show()

## Check: a cointegrating combination is stationary

Any `b` with `b @ loadings = 0` removes the factors, leaving `b @ (log p - log p0) = b @ s`.

In [ ]:
B = np.asarray(LOADINGS, dtype=float)
b = np.linalg.svd(B.T)[2][-1]  # a vector in the null space of loadings'
log_prices = fm.to_log_prices(log_returns)
logp = log_prices.to_numpy()
combo = (logp - logp[0]) @ b
print("b =", np.round(b, 4), " b @ loadings =", np.round(b @ B, 12))
ax = plt.figure(figsize=(10, 2.5)).gca()
ax.plot(log_prices.index, combo, lw=0.8)
ax.set_title("b @ (log p - log p0)")
plt.show()